# PyTorch Basics

## The deep learning framework landscape

**PyTorch** (Facebook AI Research, 2016) has become the go-to framework in academic research — most papers at NeurIPS, ICML, and ICLR tend to use it. *Pros*: the dynamic computation graph makes debugging feel natural with standard Python tools; clean, Pythonic API; rich ecosystem (Hugging Face, Lightning, torchvision). *Cons*: historically not the fastest option for production deployment, though `torch.compile` (covered later in the course) is narrowing that gap.

**TensorFlow / Keras** (Google, 2015) was the dominant choice for a few years before PyTorch took over in academic research. *Pros*: solid deployment story (TFLite for mobile, TF.js for browsers, TF Serving for production); the Keras API is quite approachable for beginners. *Cons*: TF 1.x relied on static graphs, which were efficient but frustrating to debug; TF 2 introduced eager execution, though by that point PyTorch had already gained a lot of ground.

**JAX** (Google DeepMind, 2018) is quite different from both: a functional, NumPy-compatible library organised around composable transforms — `jit` (XLA compilation), `grad`, `vmap`, `pmap`. *Pros*: elegant and composable design; excellent TPU support; gaining traction quickly in research. *Cons*: the purely functional style (no mutable state) takes some tome to digest and can make debugging trickier.

**Why PyTorch in this course**: it is widely used in research, the ecosystem around it is hard to beat, and its imperative style feels close to ordinary Python — which makes it a natural place to start.

## Setup

### Load the libraries

In [ ]:
import torch      # the main PyTorch module
import numpy as np  # we'll use numpy briefly to show the bridge with PyTorch
print(f"PyTorch version: {torch.__version__}")

### Choose the compute device

PyTorch can run on different hardware: a CPU, an NVIDIA GPU (via CUDA), or an Apple Silicon GPU (via the Metal Performance Shaders framework, `mps`). Working on a GPU makes training dramatically faster — matrix multiplications, which are the core operation in neural networks, can be done thousands of times faster on a GPU than on a CPU.

We create a `device` string variable so we can later move data and models to the appropriate device with a single `.to(device)` call. If no GPU is available this will silently fall back to the CPU.

* For NVIDIA GPUs: `torch.cuda.is_available()`
* For Apple Silicon (M1/M2/M3): `torch.backends.mps.is_available()`

In Google Colab you can get a free GPU via: Edit → Notebook settings → Hardware accelerator → GPU.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

## Part 1: Tensors

A **tensor** is the central data structure in PyTorch. You can think of it as a generalisation of the concepts you already know:
* A **scalar** is a 0-dimensional tensor (a single number).
* A **vector** is a 1-dimensional tensor.
* A **matrix** is a 2-dimensional tensor.
* And so on for 3D, 4D, ... tensors.

In practice, when training a neural network you will deal with tensors of all these shapes. For example, in our FashionMNIST demo (see notebook 02_MLP_fashionMNIST.ipynb):
* A single image is a 3D tensor of shape `[1, 28, 28]` (channel × height × width).
* A mini-batch of 128 images is a 4D tensor of shape `[128, 1, 28, 28]`.
* The weight matrix of a linear layer is a 2D tensor.

Tensors are very similar to NumPy arrays — in fact most operations have the exact same syntax — but with two crucial extra capabilities:
1. They can live on a GPU.
2. They can automatically track computations for backpropagation (autograd, covered in Part 2).

### 1.1 Creating tensors

The most direct way to create a tensor is from a Python list (or a list of lists for higher dimensions). There are two ways to do this and the difference matters:

* **`torch.tensor(data)`** (lowercase `t`): infers the dtype from the content of `data`. Python `int` → `torch.int64`, Python `float` → `torch.float32`. If you pass a NumPy array, the original dtype is preserved (e.g. `np.float64` stays `torch.float64`). Always makes a copy of the data.
* **`torch.Tensor(data)`** (capital `T`): this is the class constructor. It **always** produces a `float32` tensor regardless of the input type — so passing a list of integers silently gives you floats. This is the quirk: `torch.Tensor([1, 2, 3])` gives `tensor([1., 2., 3.])` of dtype `float32`, not `int64` as you might expect.

Use `torch.tensor()` as the default — it is explicit and predictable. You will see `torch.Tensor()` in older code, so it is worth knowing about, but it has no advantage over `torch.tensor()` in normal usage.

In [ ]:
## Demonstrate the dtype difference between torch.tensor() and torch.Tensor()
int_list = [1, 2, 3]   ## a plain Python list of integers

t_lower = torch.tensor(int_list)   ## lowercase: infers dtype from the data → int64
t_upper = torch.Tensor(int_list)   ## uppercase: always float32, ignores input type

print(f"torch.tensor([1, 2, 3]) → dtype: {t_lower.dtype},  values: {t_lower}")
print(f"torch.Tensor([1, 2, 3]) → dtype: {t_upper.dtype}, values: {t_upper}")
print()
print("With a list of floats, both agree (no surprise here):")
float_list = [1.0, 2.0, 3.0]
print(f"torch.tensor([1., 2., 3.]) → dtype: {torch.tensor(float_list).dtype}")
print(f"torch.Tensor([1., 2., 3.]) → dtype: {torch.Tensor(float_list).dtype}")

In [ ]:
# 1D tensor (a vector) from a Python list
v = torch.tensor([1.0, 2.0, 3.0, 4.0])
print(f"1D tensor: {v}")
print(f"  shape: {v.shape}")
print()

# 2D tensor (a matrix) from a list of lists
M = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(f"2D tensor (matrix):\n{M}")
print(f"  shape: {M.shape}")
print()

# 3D tensor from a list of lists of lists
T = torch.tensor([[[1.0, 2.0], [3.0, 4.0]],
                  [[5.0, 6.0], [7.0, 8.0]]])
print(f"3D tensor:\n{T}")
print(f"  shape: {T.shape}")

There are also several convenient constructors for common cases, which you will use constantly:

In [ ]:
## torch.zeros(shape): all zeros
z = torch.zeros(3, 4)       # 3 rows, 4 columns
print(f"zeros(3,4):\n{z}\n")

## torch.ones(shape): all ones
o = torch.ones(2, 3)
print(f"ones(2,3):\n{o}\n")

## torch.rand(shape): uniform random values in [0, 1)
r = torch.rand(2, 3)
print(f"rand(2,3):\n{r}\n")

## torch.randn(shape): samples from a standard normal distribution (mean=0, std=1)
rn = torch.randn(2, 3)
print(f"randn(2,3):\n{rn}\n")

## torch.arange(start, stop, step): like Python's range(), but returns a tensor
a = torch.arange(0, 10, 2)  # 0, 2, 4, 6, 8
print(f"arange(0,10,2): {a}")

### 1.2 Tensor attributes

Every tensor has three important attributes you should always be aware of:
* **`.dtype`**: the data type of the elements (e.g., `float32`, `int64`, `bool`). Most neural network computations use `float32`.
* **`.shape`**: the dimensions of the tensor, as a `torch.Size` object. `.size()` returns the same thing when called with no arguments; `.size(dim)` returns the size of a single dimension as an integer.
* **`.device`**: where the tensor lives (`cpu`, `cuda:0`, `mps:0`, etc.).

In [ ]:
x = torch.randn(3, 4)   ## create a 3x4 tensor of random floats

print(f"dtype:  {x.dtype}")    ## default is float32 for randn
print(f"shape:  {x.shape}")    ## torch.Size([3, 4])
print(f"size(): {x.size()}")   ## same result as .shape when called with no arguments
print(f"size(0): {x.size(0)}") ## .size(dim) returns the size of a specific dimension
print(f"device: {x.device}")   ## 'cpu' if we haven't moved it yet
print(f"number of elements: {x.numel()}")  ## 3*4 = 12

print()
## You can specify dtype at creation time:
xi = torch.zeros(2, 3, dtype=torch.int64)  ## integer tensor
print(f"int64 tensor dtype: {xi.dtype}")
print(xi)

### 1.3 Arithmetic operations and broadcasting

All standard arithmetic operations work element-wise on tensors of the **same shape**. PyTorch also supports **broadcasting**: if the shapes are compatible (one of them has size 1 along a dimension), the smaller tensor is automatically "expanded" to match the larger one. This is identical to NumPy broadcasting.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(f"a + b  = {a + b}")   ## element-wise addition
print(f"a * b  = {a * b}")   ## element-wise multiplication (NOT dot product)
print(f"a - b  = {a - b}")   ## element-wise subtraction
print(f"a / b  = {a / b}")   ## element-wise division
print(f"a ** 2 = {a ** 2}")  ## element-wise power
print()

## Broadcasting: add a scalar to all elements
print(f"a + 10 = {a + 10}")  ## 10 is broadcast to match the shape of a
print()

## Broadcasting: add a (3,1) tensor to a (3,3) tensor
row = torch.tensor([[1.0], [2.0], [3.0]])  ## shape (3,1)
mat = torch.ones(3, 3)                     ## shape (3,3)
print(f"row (shape {row.shape}) + mat (shape {mat.shape}):")
print(row + mat)  ## row is broadcast: each row of mat gets a different offset

### 1.4 Matrix multiplication

Element-wise `*` is NOT matrix multiplication. For the real thing use `@` (or equivalently `torch.matmul` — they are identical, `@` is just cleaner syntax):

$$C = A \cdot B \quad \text{where} \quad C_{ij} = \sum_k A_{ik} B_{kj}$$

For this to be valid the inner dimensions must match: `A` of shape `(m, k)` × `B` of shape `(k, n)` → `C` of shape `(m, n)`.

You will also encounter `torch.mm` (2D-only, no batching) and `torch.bmm` (3D batched, no broadcasting) in older code — `@` supersedes both.

In [ ]:
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

print(f"A @ B (matrix mul):\n{A @ B}\n")
print(f"A * B (element-wise, NOT matrix mul):\n{A * B}\n")

## @ and torch.matmul are identical; torch.mm is the older 2D-only version
print(f"All the same result? {torch.allclose(A @ B, torch.matmul(A, B)) and torch.allclose(A @ B, torch.mm(A, B))}")
print()

## @ also handles batched matmul: (B, m, k) @ (B, k, n) → (B, m, n)
batch_A = torch.randn(8, 3, 4)
batch_B = torch.randn(8, 4, 5)
print(f"Batched: {batch_A.shape} @ {batch_B.shape} → {(batch_A @ batch_B).shape}")

### 1.4b `torch.einsum`

`torch.einsum` is a compact notation for expressing any linear algebra operation in terms of index contractions. You write a string that names the indices of the inputs and the output, and PyTorch figures out what to compute. It is heavily used in research code, especially for attention mechanisms and tensor operations with unusual contractions (see later in the course).

The string format is `'input1_indices, input2_indices -> output_indices'`. Repeated indices that do **not** appear in the output are summed over (contracted). A few examples cover most of what you will encounter.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

## Dot product: sum over the single shared index i
print(f"dot product 'i,i->': {torch.einsum('i,i->', a, b)}")   ## 1*4+2*5+3*6 = 32

## Matrix-vector product: sum over k, keep i
v = torch.tensor([1.0, 2.0])
print(f"matrix-vector 'ik,k->i': {torch.einsum('ik,k->i', A, v)}")  ## same as A @ v

## Matrix multiplication: sum over the shared index j, keep i and k
print(f"matrix mul 'ij,jk->ik':\n{torch.einsum('ij,jk->ik', A, B)}")  ## same as A @ B

## Transpose: just swap the output indices
print(f"transpose 'ij->ji' :\n{torch.einsum('ij->ji', A)}")

## Batched matmul: b is the batch index (not contracted), j is contracted
batch_A = torch.randn(4, 3, 5)
batch_B = torch.randn(4, 5, 2)
print(f"batched matmul 'bij,bjk->bik': {torch.einsum('bij,bjk->bik', batch_A, batch_B).shape}")

### 1.5 Indexing and slicing

Tensor indexing uses exactly the same syntax as NumPy. You can use integer indices, slices (`start:stop:step`), and combinations of both.

In [ ]:
x = torch.arange(12, dtype=torch.float32).reshape(3, 4)  ## 3x4 matrix
print(f"x:\n{x}\n")

print(f"x[0]       = {x[0]}")          ## first row (a 1D tensor)
print(f"x[0, 2]    = {x[0, 2]}")       ## element at row 0, col 2
print(f"x[:, 1]    = {x[:, 1]}")       ## all rows, column 1 (second column)
print(f"x[1:, 2:]  =\n{x[1:, 2:]}")   ## rows 1 onwards, columns 2 onwards
print(f"x[0, -1]   = {x[0, -1]}")      ## last element of first row (negative index)

### 1.6 Reshaping tensors

You will reshape tensors constantly. The two main methods are `.view()` and `.reshape()`.

* **`.view()`**: returns a new tensor sharing the same underlying memory with a different shape. Only works if the tensor is contiguous in memory (usually it is). Very efficient.
* **`.reshape()`**: does the same, but can make a copy if necessary (so it always works, slightly less efficient).

A very common pattern is using `-1` as one of the dimensions: this tells PyTorch to infer that dimension automatically so that the total number of elements stays the same. For example, a batch of images with shape `[batch, C, H, W]` can be flattened to `[batch, C*H*W]` with `.view(batch, -1)` or `.reshape(batch, -1)`.

In [ ]:
x = torch.arange(24, dtype=torch.float32)   ## 1D tensor, 24 elements
print(f"Original shape: {x.shape}")
print()

## Reshape to 4 rows x 6 columns
y = x.view(4, 6)
print(f"After .view(4, 6): shape = {y.shape}")
print(y)
print()

## The -1 trick: let PyTorch infer the missing dimension
z = x.view(6, -1)    ## 6 rows, how many columns? 24/6 = 4
print(f"After .view(6, -1): shape = {z.shape}")
print()

## Practical example: flattening a batch of images
batch = torch.randn(128, 1, 28, 28)     ## 128 grayscale images of size 28x28
flat  = batch.view(128, -1)             ## flatten each image into a vector
print(f"Batch shape:   {batch.shape}")
print(f"Flattened:     {flat.shape}")
print(f"(28*28 = {28*28})")             ## the vector size is 784

### 1.7 Moving tensors to a device

To use a GPU (or MPS), tensors must be explicitly moved there. You do this with the `.to(device)` method. Note: **both operands of any operation must be on the same device**, otherwise you get an error.

In [ ]:
x_cpu = torch.tensor([1.0, 2.0, 3.0])    ## lives on CPU by default
print(f"x_cpu device: {x_cpu.device}")

## Move to the device we chose earlier (GPU if available, CPU otherwise)
x_dev = x_cpu.to(device)
print(f"x_dev device: {x_dev.device}")
print(f"x_dev values: {x_dev}")           ## same values, different device
print()

## You can also create a tensor directly on a device:
y_dev = torch.ones(3, device=device)
print(f"y_dev: {y_dev} (device: {y_dev.device})")
print()

## Operations work as normal:
print(f"x_dev + y_dev = {x_dev + y_dev}")
print()

## To bring a tensor back to CPU (e.g. for numpy conversion):
x_back = x_dev.cpu()
print(f"Back on CPU: {x_back} (device: {x_back.device})")

In [ ]:
## What happens if you try to operate on tensors that live on different devices?
## Let's force a CPU tensor and a GPU tensor into the same operation and see.
if device != "cpu":
    a_cpu = torch.tensor([1.0, 2.0, 3.0])           ## lives on CPU
    b_gpu = torch.tensor([4.0, 5.0, 6.0]).to(device) ## lives on GPU

    print(f"a_cpu device: {a_cpu.device}")
    print(f"b_gpu device: {b_gpu.device}")
    print()
    try:
        result = a_cpu + b_gpu   ## this will raise a RuntimeError
    except RuntimeError as e:
        print(f"RuntimeError: {e}")
    print()
    print("Fix: move both tensors to the same device before operating on them.")
    print(f"  (a_cpu.to(device) + b_gpu).device = {(a_cpu.to(device) + b_gpu).device}")
else:
    print("Both tensors are on CPU in this run, so no cross-device error can occur here.")
    print("Run this on a machine with a GPU (or in Google Colab) to see the RuntimeError.")

### 1.7b How much faster is a GPU? A concrete experiment

We claimed that GPUs are dramatically faster for neural network computations. Let's verify this with a simple experiment: multiply two large matrices together, first on the CPU and then on the GPU.

**Why is a GPU faster?** A CPU is optimised for low-latency, sequential tasks: it has a small number of very powerful cores (typically 8–32), each capable of handling complex, branchy code. A GPU is optimised for *throughput*: it has thousands of simpler cores that all work in parallel. Matrix multiplication is a perfectly parallel task — each element of the result `C[i,j]` can be computed independently — so a GPU can compute thousands of them simultaneously.

In [ ]:
import time

def sync(target_device):
    """Wait for all pending ops on target_device to complete before reading the clock."""
    if target_device == "cuda":
        torch.cuda.synchronize()
    elif target_device == "mps":
        torch.mps.synchronize()

def time_matmul(size, target_device, n_repeats=20):
    """
    Time n_repeats matrix multiplications of (size x size) matrices on target_device.
    Returns the median time in milliseconds.
    """
    ## Create two random matrices directly on the target device
    A = torch.randn(size, size, device=target_device)
    B = torch.randn(size, size, device=target_device)

    ## Warm-up: run a few iterations so the GPU is fully initialised
    ## (the first call often includes JIT compilation / library loading overhead)
    for _ in range(3):
        _ = A @ B

    ## Synchronise before starting the timer so the GPU has finished all pending work
    sync(target_device)

    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        C  = A @ B
        ## Synchronise again before stopping the timer:
        ## both CUDA and MPS dispatch ops asynchronously — without this,
        ## we'd stop the clock before the GPU is actually done
        sync(target_device)
        times.append((time.perf_counter() - t0) * 1000)  ## convert to milliseconds

    import statistics
    return statistics.median(times)   ## median is more robust than mean to outliers

## Matrix sizes to test: from small to large
sizes = [256, 1024, 2048, 4096]

print(f"{'Size':>6}  {'CPU (ms)':>10}  {'GPU (ms)':>10}  {'Speedup':>10}")
print("-" * 44)

for sz in sizes:
    cpu_ms = time_matmul(sz, "cpu")

    if device == "cpu":
        print(f"{sz:>6}  {cpu_ms:>10.2f}  {'  N/A':>10}  {'  N/A':>10}")
    else:
        gpu_ms  = time_matmul(sz, device)
        speedup = cpu_ms / gpu_ms
        print(f"{sz:>6}  {cpu_ms:>10.2f}  {gpu_ms:>10.2f}  {speedup:>9.1f}x")

if device == "cpu":
    print()
    print("No GPU detected. To see the speedup, run this in Google Colab with a GPU runtime")
    print("(Runtime → Change runtime type → Hardware accelerator → GPU).")

### 1.8 The NumPy bridge

PyTorch tensors on the CPU can be converted to NumPy arrays and vice versa. The crucial thing to know: **they share the same underlying memory**. Modifying one will modify the other. This is efficient (no data is copied) but can cause subtle bugs if you are not aware of it.

In [ ]:
## Tensor → NumPy: use .numpy() method (tensor must be on CPU)
t = torch.tensor([1.0, 2.0, 3.0])
n = t.numpy()    ## converts to numpy array; shares memory!
print(f"t = {t}")
print(f"n = {n}")
print()

## Modifying the tensor also modifies the numpy array (shared memory!)
t.add_(10)       ## in-place addition (methods ending in _ are in-place in PyTorch)
print("After t.add_(10):")
print(f"  t = {t}")
print(f"  n = {n}")   ## n changed too!
print()

## NumPy → Tensor: use torch.from_numpy()
array = np.array([4.0, 5.0, 6.0])
tensor_from_np = torch.from_numpy(array)   ## again, shared memory
print(f"array         = {array}")
print(f"tensor_from_np = {tensor_from_np}")
print()

np.add(array, 1, out=array)   ## in-place modification of numpy array
print("After np.add(array, 1):")
print(f"  array          = {array}")
print(f"  tensor_from_np = {tensor_from_np}")  ## tensor changed too!

## Part 2: Autograd — Automatic Differentiation

Training a neural network means repeatedly computing the **gradient** of the loss with respect to every single parameter, so we can take a small step in the direction that decreases the loss (gradient descent). A typical model has millions of parameters. Computing all those gradients by hand would be completely impractical.

PyTorch solves this with **autograd**: a system that automatically computes gradients for any computation you write. The key idea is that PyTorch builds a **computational graph** as you perform operations on tensors with `requires_grad=True`. When you call `.backward()` on a scalar output (the loss), PyTorch walks backwards through this graph, applying the chain rule, and stores the gradient in the `.grad` attribute of each leaf tensor.

### 2.1 The `requires_grad` flag

By default, tensors do **not** track gradients — this saves memory. You opt into gradient tracking by setting `requires_grad=True`. Model parameters (created inside `nn.Linear`, etc., see later in the course) have this flag set automatically.

In [ ]:
## A regular tensor: does not track gradients
a = torch.tensor([3.0, 4.0])
print(f"a.requires_grad = {a.requires_grad}")
print()

## A tensor with gradient tracking enabled
x = torch.tensor([4.0], requires_grad=True)
print(f"x = {x}")
print(f"x.requires_grad = {x.requires_grad}")
print()

## Any tensor computed FROM x also tracks gradients
y = x ** 2          ## y = x^2
print(f"y = {y}")
print(f"y.grad_fn = {y.grad_fn}")   ## grad_fn records how y was computed

### 2.2 Computing gradients with `.backward()`

Let's compute the gradient of a polynomial function and verify it matches the analytical derivative.

Let:
$$y = 3x^2 + 2x + 1$$

The derivative is:
$$\frac{dy}{dx} = 6x + 2$$

At $x = 4$: $\frac{dy}{dx} = 6 \cdot 4 + 2 = 26$.

Let's check that PyTorch gets this right.

In [ ]:
x = torch.tensor([4.0], requires_grad=True)  ## our input, at x=4; note it must be a float

## Forward pass: compute y = 3x^2 + 2x + 1
## PyTorch records every operation in the computational graph
y = 3 * x**2 + 2 * x + 1
print(f"x = {x.item():.1f}")
print(f"y = 3*x^2 + 2*x + 1 = {y.item():.1f}")
print()

## Backward pass: compute dy/dx by walking back through the graph
## .backward() requires the output to be a scalar (a single number)
y.backward()
print(f"dy/dx at x=4, computed by PyTorch:  {x.grad.item()}")
print(f"dy/dx at x=4, computed analytically: {6*4 + 2}")

### 2.3 The computational graph

When you compute `y = 3 * x**2 + 2 * x + 1`, PyTorch builds a graph of operations: each tensor resulting from an operation stores a `grad_fn` attribute that points to the operation that created it. When you call `.backward()`, PyTorch walks this graph in reverse (from `y` back to `x`), applying the chain rule at each step.

Tensors that are created directly (like `x` above) are called **leaf tensors**. Their `.grad` attribute is filled by `.backward()`. Intermediate tensors (like the result of `x**2`) are not leaf tensors and their gradients are not retained by default (to save memory).

In [ ]:
x   = torch.tensor([2.0], requires_grad=True)
tmp = x ** 2              ## intermediate result
y   = 3 * tmp + 1         ## final result

print(f"x.grad_fn   = {x.grad_fn}")    ## None: x is a leaf (created directly)
print(f"tmp.grad_fn = {tmp.grad_fn}")  ## PowBackward0: records the squaring operation
print(f"y.grad_fn   = {y.grad_fn}")    ## AddBackward0: records the +1 operation
print()

## Is it a leaf tensor?
print(f"x.is_leaf   = {x.is_leaf}")
print(f"tmp.is_leaf = {tmp.is_leaf}")
print(f"y.is_leaf   = {y.is_leaf}")

### 2.4 Disabling gradient tracking with `torch.no_grad()`

Building the computational graph has a cost: it uses extra memory and time. When we are **not** training — e.g., when evaluating the model on a test set or making predictions — we don't need gradients. The `torch.no_grad()` context manager disables graph construction within its block.

This is important in two situations:
1. **Testing/inference**: don't waste memory on gradients you don't need.
2. **Updating parameters manually**: if you try to update a parameter tensor inside its own computation graph, PyTorch would try to track that operation too, which you don't want.

In [ ]:
x = torch.tensor([3.0], requires_grad=True)

## Outside no_grad: operations ARE tracked
y = x ** 2
print(f"Outside no_grad: y.requires_grad = {y.requires_grad}")
print(f"  y.grad_fn = {y.grad_fn}")
print()

## Inside no_grad: operations are NOT tracked
with torch.no_grad():
    z = x ** 2
    print(f"Inside no_grad: z.requires_grad = {z.requires_grad}")
    print(f"  z.grad_fn = {z.grad_fn}")

### 2.5 Gradient accumulation and `zero_grad()`

There is one subtle but critical behaviour: **every call to `.backward()` accumulates gradients**, it does not replace them. That is, `x.grad += new_grad` rather than `x.grad = new_grad`.

This is intentional (there are advanced use cases where accumulation is useful), but in a standard training loop you must **reset the gradients to zero before each backward pass**. In the MLP_demo notebook you will see `optimizer.zero_grad()` — this is what that does.

Let's demonstrate the accumulation problem and the fix:

In [ ]:
x = torch.tensor([2.0], requires_grad=True)

## First backward pass
y = x ** 2      ## y = x^2, dy/dx = 2x = 4 at x=2
y.backward()
print(f"After 1st backward: x.grad = {x.grad.item()}")   ## expected: 4

## Second backward pass WITHOUT zeroing first
y = x ** 2
y.backward()
print(f"After 2nd backward (no zero): x.grad = {x.grad.item()}")  ## expected: 4, but got 8!
## The gradient was ACCUMULATED (4 + 4 = 8), not replaced!
print()

## The fix: zero the gradient before each backward
x = torch.tensor([2.0], requires_grad=True)   ## fresh tensor

y = x ** 2
y.backward()
print(f"1st backward: x.grad = {x.grad.item()}")         ## 4

x.grad.zero_()    ## reset gradient to zero in-place
y = x ** 2
y.backward()
print(f"2nd backward (after zero): x.grad = {x.grad.item()}")  ## 4 (correct!)
print()
print("In a training loop, the optimizer calls this for us: optimizer.zero_grad()")